# Entrega 3: Modelo de Datos, Métricas y Preprocesamiento

**Proyecto:** Dinámica del comercio mundial
**Objetivo:** Definir y validar la estructura relacional (Modelado de Datos) óptima para consumir la información en Tableau, evaluando opciones de esquema (Tabla Plana vs. Esquema en Estrella) para asegurar la integridad de las métricas (evitar duplicaciones) y optimizar el rendimiento.

## 1. Importación y Carga del Dataset Consolidado

In [15]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [16]:
# Asumimos la existencia del dataset limpio de la entrega 2
try:
    df = pd.read_csv('../data/processed/dataset_limpio_entrega2_consolidado.csv')
    print(f'Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.')
except FileNotFoundError:
    print('El archivo no se encontró. Generando dummy para demostración del pipeline de modelado relacional.')
    np.random.seed(42)
    years = np.repeat(np.arange(1989, 2022), 20)
    countries = np.tile([f'Country_{i}' for i in range(20)], 33)
    df = pd.DataFrame({
        'Year': years,
        'Partner Name': countries,
        'World Growth (%)': np.repeat(np.random.normal(3, 1, 33), 20),
        'Export (US$ Million)': np.random.uniform(10, 1000, 660),
        'Import (US$ Million)': np.random.uniform(10, 1000, 660),
        'Trade Status': np.random.choice(['Superávit', 'Déficit'], 660)
    })

Dataset cargado con 7783 filas y 38 columnas.


## 2. Validación de Cardinalidad y Granularidad

Antes de decidir el modelo para Tableau, verificamos la granularidad de la tabla base.

In [17]:
# La clave primaria (PK) teórica es Partner Name + Year
is_unique = df.set_index(['Partner Name', 'Year']).index.is_unique
print(f'¿Es la combinación País-Año única por fila?: {is_unique}')

total_exports_base = df['Export (US$ Million)'].sum()
print(f'Total de exportaciones base: {total_exports_base:,.2f}')

¿Es la combinación País-Año única por fila?: True
Total de exportaciones base: 391,081,394.11


## 3. Preprocesamiento: Opciones de Modelado

### Opción 1: Modelo de Tabla Plana (One Big Table / OBT)
En este modelo, todas las dimensiones (País, Año, Macroeconomía) conviven en la misma tabla de hechos. Es la estructura actual del `df`.

In [18]:
df_flat = df.copy()
mem_flat = df_flat.memory_usage(deep=True).sum() / 1024**2
print(f'Memoria de la Tabla Plana: {mem_flat:.2f} MB')
print(f'Columnas redundantes (ej. World Growth se repite por país en el mismo año).')

Memoria de la Tabla Plana: 3.35 MB
Columnas redundantes (ej. World Growth se repite por país en el mismo año).


### Opción 2: Modelo en Estrella (Star Schema)
Separamos las variables descriptivas en tablas de Dimensiones (`Dim_...`) y mantenemos solo llaves primarias y métricas en la tabla de Hechos (`Fact_...`).

In [19]:
# 1. Dimensión Tiempo y Macroeconomía (Dim_Time)
# Contiene variables que solo dependen del año, como el crecimiento global.
if 'World Growth (%)' in df.columns:
    dim_time = df[['Year', 'World Growth (%)']].drop_duplicates().reset_index(drop=True)
else:
    dim_time = df[['Year']].drop_duplicates()
print(f'Dim_Time: {dim_time.shape[0]} filas, {dim_time.shape[1]} columnas')

# 2. Dimensión Geográfica (Dim_Country)
dim_country = df[['Partner Name']].drop_duplicates().reset_index(drop=True)
# En un caso real aquí agregaríamos latitud, longitud, ISO Code, Región, etc.
print(f'Dim_Country: {dim_country.shape[0]} filas, {dim_country.shape[1]} columnas')

# 3. Tabla de Hechos (Fact_Trade)
# Quitamos columnas que ya están en las dimensiones (excepto las llaves foráneas: Year, Partner Name)
cols_to_drop_from_fact = [c for c in ['World Growth (%)'] if c in df.columns]
fact_trade = df.drop(columns=cols_to_drop_from_fact)
print(f'Fact_Trade: {fact_trade.shape[0]} filas, {fact_trade.shape[1]} columnas')

Dim_Time: 34 filas, 2 columnas
Dim_Country: 252 filas, 1 columnas
Fact_Trade: 7783 filas, 37 columnas


## 4. Métricas de Evaluación y Criterios de Comparación

Comparamos la integridad estructural de ambos modelos.

In [20]:
# 1. Validación de Totales (Evitar Fan-out)
total_exports_fact = fact_trade['Export (US$ Million)'].sum()
print(f'Total Export (Star Schema): {total_exports_fact:,.2f}')
print(f'¿Se mantiene la integridad de los totales numéricos?: {np.isclose(total_exports_base, total_exports_fact)}')

# 2. Evaluación de Almacenamiento Lógico
mem_star = (dim_time.memory_usage(deep=True).sum() + 
            dim_country.memory_usage(deep=True).sum() + 
            fact_trade.memory_usage(deep=True).sum()) / 1024**2

print(f'Memoria Total Esquema en Estrella: {mem_star:.2f} MB')
print(f'Diferencia de memoria: {mem_flat - mem_star:.2f} MB (A escala mayor, la estrella es más eficiente)')

# 3. Riesgo de cálculo sobre 'World Growth (%)'
# En la tabla plana, si promediamos el World Growth, se sesgará si hay años con más países registrados.
mean_growth_flat = df['World Growth (%)'].mean()
mean_growth_dim = dim_time['World Growth (%)'].mean()
print(f'Promedio de crecimiento global (Tabla Plana - Sesgado): {mean_growth_flat:.2f}%')
print(f'Promedio de crecimiento global (Dimensión - Correcto): {mean_growth_dim:.2f}%')

Total Export (Star Schema): 391,081,394.11
¿Se mantiene la integridad de los totales numéricos?: True
Memoria Total Esquema en Estrella: 3.31 MB
Diferencia de memoria: 0.04 MB (A escala mayor, la estrella es más eficiente)
Promedio de crecimiento global (Tabla Plana - Sesgado): 1.89%
Promedio de crecimiento global (Dimensión - Correcto): 1.96%


## 5. Exportación de Fuentes Finales para Tableau

Basado en las métricas, procedemos a exportar las tablas separadas para utilizar la capa lógica (Relationships) de Tableau.

In [21]:
import os
os.makedirs('../outputs/tableau_sources', exist_ok=True)

# Exportar dimensiones y hechos para Tableau
fact_trade.to_csv('../outputs/tableau_sources/Fact_Trade.csv', index=False)
dim_time.to_csv('../outputs/tableau_sources/Dim_Time.csv', index=False)
dim_country.to_csv('../outputs/tableau_sources/Dim_Country.csv', index=False)

print('Archivos exportados en /outputs/tableau_sources/ listos para ser conectados mediante Relationships en Tableau.')

Archivos exportados en /outputs/tableau_sources/ listos para ser conectados mediante Relationships en Tableau.
